# YAML - Rust

All 5 Rust examples from [docs/yaml.md](https://platob.github.io/yggdryl/yaml/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and expect the
[evcxr](https://github.com/evcxr/evcxr) kernel. Declare the crate once, in
a cell of your own, before running them:

```rust
:dep yggdryl = { version = "0.1", features = ["parquet", "iceberg"] }
```

## Raw shared-Scalar access

In [ ]:
use yggdryl::{yaml, Scalar};

let value = yaml::from_utf8("symbol: AAPL\nquantity: 2\n")?;

assert_eq!(
    value.get_key_str("symbol").and_then(Scalar::as_utf8),
    Some("AAPL")
);
assert_eq!(yaml::into_utf8(&value)?, "quantity: 2\nsymbol: AAPL\n");

## Natural values and exact Fields

In [ ]:
use yggdryl::{yaml, DataType, Field, Scalar};

let amount = Field::new("amount", DataType::decimal128(8, 2)?, false);
let decoded = yaml::from_utf8_with_field("'12.50'\n", &amount)?;

assert_eq!(decoded, Scalar::d128(1_250, 2));

## Documents and streams

In [ ]:
use yggdryl::yaml;

let documents = yaml::from_utf8_all("id: 1\n---\nid: 2\n")?;
let mut destination = Vec::new();
yaml::into_writer_all(&documents, &mut destination)?;

assert_eq!(documents.len(), 2);
assert_eq!(yaml::from_bytes_all(&destination)?, documents);

## Formatting

In [ ]:
use yggdryl::text::Formatting;
use yggdryl::{yaml, Scalar};

let value = Scalar::from_record([("id", Scalar::I64(1))])?;
let flow =
    yaml::into_utf8_with_formatting(&value, Formatting::compact())?;

assert_eq!(flow, "{id: 1}\n");
assert_eq!(yaml::from_utf8(&flow)?, value);

## Placeholders

In [ ]:
use yggdryl::text::{Format, Loading, Placeholders};
use yggdryl::Scalar;

let loading = Loading::new().with_placeholders(
    Placeholders::new().with_variable("PORT", Scalar::I64(8080)),
);
let value = yggdryl::text::from_utf8_with(
    "port: \"{{ PORT }}\"\n",
    Format::Yaml,
    &loading,
)?;

assert_eq!(value.get_key_str("port"), Some(&Scalar::I64(8080)));